# Corpus Cleaning Jupyter Notebook
This Notebook produces three hypothesis-specific datasets and not a single generic cleaned corpus. 

## 0. Setup and Imports

Cell 1 imports the required libraries for data handling and Natural Language Processing, downloads NLTK resources, loads the processed full corpus and hypothesis-specific datasets created in Notebook 02, and initiliazes English stopwords and a lemmatizer for later text processing. 

In [1]:
import pandas as pd
import os
import numpy as np
import json
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import pickle
import re

# Download NLTK resources
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')


# Define base path
base_path = os.path.expanduser('~/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed')

# Load full corpus
full_corpus_path = os.path.join(base_path, 'full_corpus.json')
with open(full_corpus_path, 'r', encoding='utf-8') as f:
    full_corpus = json.load(f)

def load_text_folder(folder_path):
    data = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith('.txt'):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                data.append({'filename': filename, 'text': f.read()})
    return data

h1_dataset = load_text_folder(os.path.join(base_path, 'h1'))
h2_dataset = load_text_folder(os.path.join(base_path, 'h2'))
h3_dataset = load_text_folder(os.path.join(base_path, 'h3'))

# Initialize NLP tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/sophiehamann/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [2]:
year_map = {"03": 1977, "04": 1978, "06": 1978, "07": 1979, "08": 1979, "09": 1980, "10": 1980, "11": 1981, "12": 1981}
for record in full_corpus:
    if record["issue"] in year_map:
        record["year"] = year_map[record["issue"]]

## 1. Hypothesis 1 - Lexical Corpus Construction

**Hypothesis 1: Conceptual Shifts**

Quantitative analysis of terminology and key concepts in Heresies will reveal distinct temporal shifts corresponding to major phases in feminist theory, including essentialist feminism in the late 1970s, the rise of French and difference feminism in the 1980s, and increasingly deconstructive theoretical orientations by the early 1990s. (Testable with: n-grams and keyword analysis.)

- Needed cleaning steps: Tokenization, (Artifact filtering), Lowercasing, Lemmatization, Stopword handling 

#### 1.1 Input Preparation

Cell 2 filters the full corpus for Hypothesis 1 documents, verifies region distribution per issue, computes basic structural statistics (regions, characters, average length), and prepares the H1 corpus for subsequent text cleaning.

In [3]:
# H1: Prepare corpus from XML files
from lxml import etree
from pathlib import Path

h1_corpus = []

for xml_file in sorted((Path(base_path) / "h1").glob("*.xml")):
    tree = etree.parse(str(xml_file))
    root = tree.getroot()
    
    issue = root.get("number")
    year = root.get("year")
    year = int(year) if year and year != "unknown" else None
    
    # Extract all text content from paragraphs
    paragraphs = [elem.text for elem in root.findall(".//paragraph") if elem.text]
    
    h1_corpus.append({
        "issue": issue,
        "year": year,
        "text": "\n\n".join(paragraphs)
    })

# Basic structural statistics
total_docs = len(h1_corpus)
char_lengths = [len(doc.get('text', '')) for doc in h1_corpus]

print(f"H1 Corpus Statistics:")
print(f"Total documents: {total_docs}")
print(f"Average text length: {np.mean(char_lengths):.0f} characters")
print(f"Min/Max text length: {min(char_lengths)}/{max(char_lengths)} characters")

h1_corpus_prepared = h1_corpus

H1 Corpus Statistics:
Total documents: 11
Average text length: 450426 characters
Min/Max text length: 343806/609104 characters


#### 1.2 Tokenization and Lowercasing
[Information on word_tokenize()](https://www.geeksforgeeks.org/nlp/tokenize-text-using-nltk-python/)

Adjusted the following cell with this [documentation](https://kristopherkyle.github.io/corpus-analysis-python/Python_Tutorial_4.html) 

Cell 3 defines and applies NLTK´s word_tokenize() to segment H1 text into word-level tokens, validates behavior on sample and edge cases (hyphenation, punctuation, contractions), and confirms readiness for lemmatization and stopword filtering. 

In this case, lowercasing does not encode conceptual difference for Hypothesis 1. It is necessary for a reliable type frequency comparison. 

In [4]:
# Tokenize H1 corpus
def tokenize_text(text):
    text = text.lower()
    return word_tokenize(text)

# Apply tokenization to H1 corpus
for doc in h1_corpus_prepared:
    doc['tokens'] = tokenize_text(doc.get('text', ''))

# Validate on sample documents
print("Tokenization sample (first 3 documents):\n")
for i, doc in enumerate(h1_corpus_prepared[:3]):
    tokens = doc['tokens'][:20]
    print(f"Doc {i}: {tokens}")
    print(f"Total tokens: {len(doc['tokens'])}\n")

# Test edge cases
test_cases = ["don't", "self-awareness", "1970s", "co-authored"]
print("Edge case handling:")
for case in test_cases:
    print(f"{case} -> {tokenize_text(case)}")


# Show before/after lowercasing sample
print("\nBefore lowercasing (sample):")
print(h1_corpus_prepared[0]['tokens'][:15])
print("\nAfter lowercasing (sample):")
print(h1_corpus_prepared[0]['tokens'][:15])

Tokenization sample (first 3 documents):

Doc 0: ['heresies', 'is', 'an', 'idea-oriented', 'journal', 'devoted', 'to', 'the', 'examination', 'of', 'art', 'and', 'politics', 'from', 'a', 'feminist', 'perspective', '.', 'we', 'believe']
Total tokens: 67509

Doc 1: ['heresies', 'is', 'an', 'idea-oriented', 'journal', 'devoted', 'to', 'the', 'examination', 'of', 'art', 'and', 'politics', 'from', 'a', 'feminist', 'perspective', '.', 'we', 'believe']
Total tokens: 96924

Doc 2: ['this', 'issue', 'of', 'heresies', 'was', 'edited', ',', 'designed', ',', 'and', 'put', 'together', 'by', 'lesbians', ',', 'four', 'of', 'whom', 'are', 'members']
Total tokens: 79600

Edge case handling:
don't -> ['do', "n't"]
self-awareness -> ['self-awareness']
1970s -> ['1970s']
co-authored -> ['co-authored']

Before lowercasing (sample):
['heresies', 'is', 'an', 'idea-oriented', 'journal', 'devoted', 'to', 'the', 'examination', 'of', 'art', 'and', 'politics', 'from', 'a']

After lowercasing (sample):
['heresies',

##### Remove punctuation 

In [5]:
# Remove punctuation tokens
import string
print("These punctuation types are being removed:",string.punctuation)
punctuation = set(string.punctuation)

for doc in h1_corpus_prepared:
    doc['tokens'] = [token for token in doc['tokens'] if token not in punctuation]

These punctuation types are being removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


#### (1.3 OCR Artifact Filtering) (May not be needed at all)
- It should report tokens removed per issue
- Spot-checking removals to confirm no theoretically significant vocabulary lost

#### 1.4 Lemmatization


Applies WordNet lemmatization to normalize tokens morphologically, consolidating inflected variants (e.g., woman/women, feminist/feminists) to prevent type inflation and enable reliable longitudinal concept tracking. A before/after sample of key feminist theoretical terms illustrates the normalization effect.

In [6]:
# Show before/after on feminist theoretical terms
feminist_terms = [token for doc in h1_corpus_prepared for token in doc['tokens'] 
                   if any(term in token for term in ['woman', 'feminist', 'essential', 'differ', 'gender', 'patriarch'])]
feminist_sample = list(set(feminist_terms))[:10]

print("Before lemmatization (sample feminist terms):")
print(feminist_sample)

# Apply lemmatization
for doc in h1_corpus_prepared:
    doc['tokens_lemmatized'] = [lemmatizer.lemmatize(token) for token in doc['tokens']]

print("\nAfter lemmatization:")
for term in feminist_sample:
    lemmatized = lemmatizer.lemmatize(term)
    print(f"{term} -> {lemmatized}")

Before lemmatization (sample feminist terms):
['essentialness', 'womanwith', 'woman.38', 'woman/bed', 'superwoman', 'feministcollective', 'patriarchal-capitalist', 'differed', 'all-woman', 'gender-based']

After lemmatization:
essentialness -> essentialness
womanwith -> womanwith
woman.38 -> woman.38
woman/bed -> woman/bed
superwoman -> superwoman
feministcollective -> feministcollective
patriarchal-capitalist -> patriarchal-capitalist
differed -> differed
all-woman -> all-woman
gender-based -> gender-based


#### 1.5 Stopword Handling
- Loading the stopword list and display which theoretically loaded words it would remove (body, difference, identity, other, work, self, etc.)
- It is important to define a custom stopword list (with purely functional words only)
- Document every word included in your custom list and the rationale
- Apply and report how many tokens were removed

##### NLTK Stopword Choice Justification
[NLTK Python Documentatoin](https://pythonspot.com/nltk-stop-words/)

[NLTK Official Documentation](https://www.nltk.org/)


It was decided to use NLTK´s stopword corpus, because it offers a transparent, customizable foundation for domain-specific text-processing. It allows direct inspection and modification of stopwords lists. Especially the modification is very important with regards to Heresies magazine, as language is inherently important for feminist theory of the second wave. 
The feminist theoretical vocabulary often overlaps with grammatical function words. With NLTK it is possible to preserve crucial terms while removing only purely functional verbs (articles, prepositions, auxiliary verbs).

In comparison to spaCy´s stopwords handling, it is similarly customizable, but tightly integrated with its full NLP pipeline. For this thesis project, these additional components are unnecessary NLTK's modular approach better suits selective application of individual processing steps.

In [7]:
# Layer A: Standard function words
layer_a = {
    'the', 'a', 'an', 'is', 'was', 'were', 'be', 'been', 'being',
    'to', 'of', 'in', 'at', 'on', 'for', 'from', 'up', 'about', 'into',
    'and', 'but', 'or', 'nor', 'as', 'if', 'because',
    'with', 'by', 'through', 'during', 'before', 'after',
    'have', 'has', 'had', 'do', 'does', 'did',
    'can', 'could', 'would', 'should', 'may', 'might', 'must'
}

# Layer B: Corpus-specific noise words
layer_b = {
    'issue', 'volume', 'editor', 'editorial', 'collective', 'page',
    'magazine', 'copyright', 'reprinted', 'published', 'heresies'
}

# Layer C: Words to REMOVE from NLTK stopwords (theoretically important)
layer_c = {
    'against', 'between', 'above', 'below', 'own', 'same', 'other',
    'most', 'few', 'each', 'herself', 'himself', 'themselves',
    'who', 'whom', 'before', 'after'
}

# Build custom stopwords
custom_stopwords = (layer_a | layer_b) - layer_c

print("Layer A (function words):", len(layer_a), "words")
print("Layer B (corpus noise):", len(layer_b), "words")
print("Layer C (removed from NLTK):", len(layer_c), "words")
print(f"\nCustom stopwords total: {len(custom_stopwords)} words")

# Apply stopword filtering
for doc in h1_corpus_prepared:
    before = len(doc['tokens_lemmatized'])
    doc['tokens_cleaned'] = [token for token in doc['tokens_lemmatized'] if token not in custom_stopwords]
    after = len(doc['tokens_cleaned'])
    doc['tokens_removed'] = before - after

# Report removal statistics
total_before = sum(len(doc['tokens_lemmatized']) for doc in h1_corpus_prepared)
total_after = sum(len(doc['tokens_cleaned']) for doc in h1_corpus_prepared)
total_removed = total_before - total_after

print(f"\nStopword removal statistics:")
print(f"Total tokens before: {total_before}")
print(f"Total tokens after: {total_after}")
print(f"Total tokens removed: {total_removed} ({100*total_removed/total_before:.1f}%)")

Layer A (function words): 45 words
Layer B (corpus noise): 11 words
Layer C (removed from NLTK): 17 words

Custom stopwords total: 54 words

Stopword removal statistics:
Total tokens before: 855855
Total tokens after: 601745
Total tokens removed: 254110 (29.7%)


#### 1.6 Output Construction

This cell constructs the final lemmatized token-level dataset per issue, aggregated tokens into defined temporal periods, and exports a token-level corpus file (corpus_h1_tokens.csv) and frequency tables per issue and per period for quantitative analysis.

In [8]:
# Map issues to volumes
def get_volume(issue):
    issue_num = int(issue)
    return (issue_num - 1) // 4 + 1

# Create lemmatized token list with metadata
token_records = []
for doc in h1_corpus_prepared:
    issue = doc.get('issue')
    volume = get_volume(issue)
    year = doc.get('year')
    for token in doc['tokens_cleaned']:
        token_records.append({
            'issue': issue,
            'volume': volume,
            'year': year,
            'token': token,
            'lemma': token
        })

# Create DataFrame
df_h1_tokens = pd.DataFrame(token_records)

# Export main token list
df_h1_tokens.to_csv('../data/processed/corpus_h1_tokens.csv', index=False)

# Frequency table per issue
# freq_per_issue = df_h1_tokens.groupby(['issue', 'year']).size().reset_index(name='frequency')
# freq_per_issue.to_csv('../data/processed/h1_frequency_per_issue.csv', index=False)

# Frequency table per volume
# freq_per_volume = df_h1_tokens.groupby('volume').size().reset_index(name='frequency')
# freq_per_volume.to_csv('../data/processed/h1_frequency_per_volume.csv', index=False)

print(f"H1 tokens exported: {len(df_h1_tokens)} rows")
print(f"Issues: {df_h1_tokens['issue'].nunique()}")
print(f"Volumes: {df_h1_tokens['volume'].nunique()}")

H1 tokens exported: 601745 rows
Issues: 11
Volumes: 3


Used this documentation:
[Click here](https://kristopherkyle.github.io/corpus-analysis-python/Python_Tutorial_4.html) 

https://github.com/kristopherkyle/corpus-analysis-python?tab=readme-ov-file

In [9]:
# Define frequency function
def freq_simple(tok_list):
    freq = {}
    for x in tok_list:
        if x not in freq:
            freq[x] = 1
        else:
            freq[x] += 1
    return freq

# Define volume mapping
def get_volume(issue):
    issue_num = int(issue)
    return (issue_num - 1) // 4 + 1

# Calculate frequency per issue
freq_per_issue = {}
for doc in h1_corpus_prepared:
    issue = doc.get('issue')
    year = doc.get('year')
    key = (issue, year)
    freq_per_issue[key] = freq_simple(doc['tokens_cleaned'])

# Convert to DataFrame
freq_issue_records = []
for (issue, year), freq_dict in freq_per_issue.items():
    for token, count in freq_dict.items():
        freq_issue_records.append({'issue': issue, 'year': year, 'token': token, 'frequency': count})

df_freq_per_issue = pd.DataFrame(freq_issue_records)
df_freq_per_issue.to_csv(os.path.join(base_path, 'h1/h1_frequency_per_issue.csv'), index=False)

# Calculate frequency per volume
freq_per_volume = {}
for doc in h1_corpus_prepared:
    volume = get_volume(doc.get('issue'))
    if volume not in freq_per_volume:
        freq_per_volume[volume] = freq_simple(doc['tokens_cleaned'])
    else:
        for token, count in freq_simple(doc['tokens_cleaned']).items():
            freq_per_volume[volume][token] = freq_per_volume[volume].get(token, 0) + count

# Convert to DataFrame
freq_vol_records = []
for volume, freq_dict in freq_per_volume.items():
    for token, count in freq_dict.items():
        freq_vol_records.append({'volume': volume, 'token': token, 'frequency': count})

df_freq_per_volume = pd.DataFrame(freq_vol_records)
df_freq_per_issue = df_freq_per_issue[df_freq_per_issue['token'] != 'paragraph']
df_freq_per_volume = df_freq_per_volume[df_freq_per_volume['token'] != 'paragraph']
df_freq_per_volume.to_csv(os.path.join(base_path, 'h1/h1_frequency_per_volume.csv'), index=False)

print(f"H1 frequency tables exported")

H1 frequency tables exported


## 2. Hypothesis 2 - Reference Dataset Construction

**Hypothesis 2: References and Influences**

References to feminist texts and thinkers in Heresies will show changing patterns of prominence over time, reflecting a move from foundational feminist works toward French feminist theory and later deconstruction-oriented influences (Uses frequency and distribution)

#### 2.1 Input Preparation

Creates a working copy of the H2 dataset, displays sample reference regions to assess structural patterns, computes basic corpus statistics (documents, total and average length), and prepares the dataset for structured extraction.

In [10]:
from lxml import etree
from pathlib import Path

# H2: Prepare corpus from XML files with separated reference lists and paragraphs
h2_corpus = []

for xml_file in sorted((Path(base_path) / "h2").glob("*.xml")):
    tree = etree.parse(str(xml_file))
    root = tree.getroot()
    
    issue = root.get("number")
    year = root.get("year")
    year = int(year) if year and year != "unknown" else None
    
    # Extract reference lists and paragraphs separately
    reference_lists = [elem.text for elem in root.findall(".//reference_list") if elem.text]
    paragraphs = [elem.text for elem in root.findall(".//paragraph") if elem.text]
    
    h2_corpus.append({
        "issue": issue,
        "year": year,
        "reference_list_text": "\n\n".join(reference_lists),
        "paragraph_text": "\n\n".join(paragraphs),
        "text": "\n\n".join(reference_lists + paragraphs)
    })

# Basic structural statistics
total_docs = len(h2_corpus)
ref_lengths = [len(doc.get('reference_list_text', '')) for doc in h2_corpus]
para_lengths = [len(doc.get('paragraph_text', '')) for doc in h2_corpus]

print(f"H2 Corpus Statistics:")
print(f"Total documents: {total_docs}")
print(f"Average reference list length: {np.mean(ref_lengths):.0f} characters")
print(f"Average paragraph length: {np.mean(para_lengths):.0f} characters")
print(f"Min/Max reference list length: {min(ref_lengths)}/{max(ref_lengths)} characters")
print(f"Min/Max paragraph length: {min(para_lengths)}/{max(para_lengths)} characters")

h2_corpus_prepared = h2_corpus

H2 Corpus Statistics:
Total documents: 11
Average reference list length: 24349 characters
Average paragraph length: 450426 characters
Min/Max reference list length: 2763/73966 characters
Min/Max paragraph length: 343806/609104 characters


#### 2.2 Reference Region Inspection to get an Overview

In [11]:
import random

# Sample 5 random issues and display their reference lists
sample_issues = random.sample(h2_corpus_prepared, min(5, len(h2_corpus_prepared)))

for doc in sample_issues:
    print("=" * 80)
    print(f"Issue {doc['issue']} | Year {doc['year']}")
    print("=" * 80)
    print(doc['reference_list_text'][:1000])  # Show first 1000 characters
    if len(doc['reference_list_text']) > 1000:
        print("\n... (truncated) ...\n")
    print()

Issue 10 | Year 1980
or concert music. It does not attempt to address the questions related to womens participation in folk, popular, or jazz music in Europe or America, or the music of non Western traditions See Gerda Lerner, "Placing Women in History. A 1975 Perspective" in Liberating Women's History, ed. Berenice A. Carroll (Urbana: University of Illinois Press, 1976) for a discusssion of the origins of the terms "women worthies," "compensatory history," or "contribution history." 3. For a survey of the literature about women in music see my Women in Music History. A Research Guide (1977) available by writing PO. Box 436 Ansonia Station, NYC, 10023 and Women in American Music, a bibliography compiled under the direction of Adrienne Fried Bloch and carol Neuls Bates to be published by Creenwood Press. Westport, Conn. 4. One meaning of reclamation is "restoration to use," a meaning that suits our purposes here. We are rediscovering this history, not for the sake of the women of the pa

#### 2.5 Reference lists: NER → normalize → count variations

##### 2.5.1 NER

Rather than normalizing before NER, because:

NER will catch the names more accurately if they're in their original form (various spellings, initials, etc.)
You'll have a complete picture of all the variation that exists
Then normalization cleans it up for analysis

In [12]:
#### 2.5.0 Load spaCy Model

import subprocess
import sys

try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    import spacy
    nlp = spacy.load("en_core_web_sm")

print("✓ spaCy model loaded: en_core_web_sm")

✓ spaCy model loaded: en_core_web_sm


In [13]:
# IMPROVED 2.5.1 - Final version with phrase and publisher filtering

print("\n#### 2.5.1 NER on Reference Lists (Fully Cleaned)\n")

# Extract named entities from reference lists
reference_list_entities = []

# Filter out: single names, publisher names, places, abbreviations
filter_out = {
    # Single letter initials
    'j', 'i', 'k', 'c', 'p', 'r', 's', 'm', 'a', 'b', 'e', 'l', 'g', 'w', 'f', 'd', 'n',
    # Common first names only (these appear without last names)
    'baltimore', 'barbara', 'ann', 'ruth', 'margaret', 'simon', 'james', 'thomas', 
    'cal', 'dutton', 'linda', 'robert', 'morris', 'marshall', 'charles',
    # Common publishers, places, abbreviations
    'press', 'university', 'new', 'york', 'london', 'paris', 'cambridge', 'chicago',
    'harper', 'yale', 'oxford', 'princeton', 'mcgraw', 'hill', 'grove', 'ed', 'eds',
    'et', 'al', 'pp', 'vol', 'no', 'trans', 'intro', 'foreword', 'preface',
    'simon', 'schuster', 'penguin', 'doubleday', 'holt', 'rinehart', 'winston',
    # Specific publishers/journals to filter
    'schocken books', 'william morrow', 'bantam books', 'sinister wisdom', 'lesbian tide',
    'w. w. norton', 'w.w. norton', 'norton',
    # Phrases/titles misidentified as people
    'female militancy',
}

# Publisher/journal keywords to filter by pattern
publisher_keywords = {'books', 'press', 'publications', 'publishers', 'journal', 'tide', 'wisdom', 'norton'}

for doc in h2_corpus_prepared:
    issue = doc['issue']
    year = doc['year']
    reference_text = doc['reference_list_text']
    
    # Process text with spaCy
    doc_processed = nlp(reference_text)
    
    # Extract PERSON entities
    for ent in doc_processed.ents:
        if ent.label_ == "PERSON":
            name = ent.text.strip()
            name_lower = name.lower()
            
            # Filter conditions:
            # 1. Skip if exactly matches filter list
            if name_lower in filter_out:
                continue
            
            # 2. Skip if it's a single word (no space = no first+last name)
            if ' ' not in name:
                continue
            
            # 3. Skip if too short overall
            if len(name) < 5:
                continue
            
            # 4. Skip if it contains publisher/publication keywords
            if any(keyword in name_lower for keyword in publisher_keywords):
                continue
            
            # 5. Skip if it contains common publisher/place patterns
            if any(pattern in name_lower for pattern in 
                   ['press', 'university', 'pub', 'edition', 'vol', 'series']):
                continue
            
            # If it passes all filters, add it
            reference_list_entities.append({
                'source': 'reference_list',
                'raw_name': name,
                'issue': issue,
                'year': year
            })

print(f"Total PERSON entities extracted from reference lists: {len(reference_list_entities)}")
print(f"\nSample extracted names (first 30):")
for i, entity in enumerate(reference_list_entities[:30]):
    print(f"  {i+1}. {entity['raw_name']}")


#### 2.5.1 NER on Reference Lists (Fully Cleaned)

Total PERSON entities extracted from reference lists: 1656

Sample extracted names (first 30):
  1. Philip Schaff et al
  2. J. Offaolain
  3. L. Martines
  4. Ancrene Riwle
  5. Ancrene Riwle
  6. E. J. Dobson
  7. G.C. Coulton
  8. La Clef
  9. Jehan de Nostredame
  10. Camille Chabaneau
  11. Litteraire des Troubadours
  12. Victor Balaguer
  13. Jehan de Nostredame
  14. Frederick Warne
  15. Remedia Amoris
  16. Leo Steinberg
  17. Thomas B. Hess
  18. Linda Nochlin
  19. Herschel B. Chipp
  20. Max Kozloff
  21. Sherry Ortner
  22. Jules Michelet
  23. La Femme
  24. J. W. Palmer
  25. Jackie MacMillan
  26. Patricia Mainardi
  27. Elizabeth Weatherford
  28. Kathryn C. Johnson
  29. Betsy Damon
  30. Carole Fisher


##### 2.5.2 Identify Spelling Variations

In [14]:
#### 2.5.2 Identify Spelling Variations

from collections import defaultdict

# Count raw name frequencies
raw_name_freq = defaultdict(int)
for entity in reference_list_entities:
    raw_name_freq[entity['raw_name']] += 1

# Display top names
sorted_names = sorted(raw_name_freq.items(), key=lambda x: x[1], reverse=True)
print(f"Top 35 most frequently extracted names in reference lists:")
print("="*70)
for i, (name, count) in enumerate(sorted_names[:35]):
    print(f"{i+1:2d}. [{count:3d}x] {name}")

Top 35 most frequently extracted names in reference lists:
 1. [  8x] Charles Seeger
 2. [  7x] Charlotte Bunch
 3. [  7x] Alan Lomax
 4. [  7x] Inez Garcia
 5. [  7x] Ruth Crawford Seeger
 6. [  6x] Sheila Rowbotham
 7. [  6x] Virginia Woolf
 8. [  6x] Romaine Brooks
 9. [  5x] Linda Nochlin
10. [  5x] Antonio Gramsci
11. [  5x] Shirley Ardener
12. [  5x] Grazyna Bacewicz
13. [  5x] Philip Johnson
14. [  4x] Elizabeth Weatherford
15. [  4x] Susana Torre
16. [  4x] Edith Thomas
17. [  4x] George Sand
18. [  4x] Gayle Rubin
19. [  4x] Rayna R. Reiter
20. [  4x] Ruth Crawford
21. [  4x] M. Rosaldo
22. [  4x] L. Lamphere
23. [  3x] Nancy Myron
24. [  3x] Louise Lamphere
25. [  3x] Emily Carr
26. [  3x] Amy Lowell
27. [  3x] Ann Sutherland
28. [  3x] H.W. Wilson
29. [  3x] Jan Oxenberg
30. [  3x] Barbara Grier
31. [  3x] Englewood Cliffs
32. [  3x] N. J.:
33. [  3x] Charles Rennie Mackintosh
34. [  3x] Kegan Paul
35. [  3x] John Wiley


##### 2.5.3 Manual Spelling Lookup Table

In [15]:
#### 2.5.3 Build Manual Spelling Lookup Table

print("\n#### 2.5.3 Build Manual Spelling Lookup Table\n")

from collections import defaultdict

# First, let's examine all variants of names that might have multiple forms
# Count raw name frequencies again
raw_name_freq = defaultdict(int)
for entity in reference_list_entities:
    raw_name_freq[entity['raw_name']] += 1

# Show all variants (sorted by frequency)
sorted_names = sorted(raw_name_freq.items(), key=lambda x: x[1], reverse=True)
print("All extracted names (sorted by frequency):")
print("="*70)
for i, (name, count) in enumerate(sorted_names):
    print(f"{i+1:3d}. [{count:2d}x] {name}")

# Now create the lookup table manually
# Map all variants to their canonical form
# For example:
# "Ruth Crawford" -> "Ruth Crawford Seeger" (more complete form)
# "de Beauvoir, Simone" -> "Simone de Beauvoir" (consistent ordering)

spelling_lookup = {
    # Ruth Crawford variants - map shorter version to fuller version
    "Ruth Crawford": "Ruth Crawford Seeger",
    
    # Add more variants as you identify them in the full list above
    # Examples of patterns to look for:
    # - "Last, First" vs "First Last" ordering
    # - Abbreviated first names vs full names
    # - Hyphenated name variations
    # - Spelling variations (e.g., British vs American)
}

print(f"\n\nSpelling lookup table created with {len(spelling_lookup)} mappings:")
for raw, canonical in sorted(spelling_lookup.items()):
    print(f"  '{raw}' → '{canonical}'")

print(f"\nNote: Review the full list above to identify additional variants.")
print(f"Common patterns to look for:")
print(f"  - Abbreviated first names (J. Smith vs James Smith)")
print(f"  - Name ordering (Last, First vs First Last)")
print(f"  - Hyphenated variations")
print(f"  - Accents/diacritics (if applicable)")


#### 2.5.3 Build Manual Spelling Lookup Table

All extracted names (sorted by frequency):
  1. [ 8x] Charles Seeger
  2. [ 7x] Charlotte Bunch
  3. [ 7x] Alan Lomax
  4. [ 7x] Inez Garcia
  5. [ 7x] Ruth Crawford Seeger
  6. [ 6x] Sheila Rowbotham
  7. [ 6x] Virginia Woolf
  8. [ 6x] Romaine Brooks
  9. [ 5x] Linda Nochlin
 10. [ 5x] Antonio Gramsci
 11. [ 5x] Shirley Ardener
 12. [ 5x] Grazyna Bacewicz
 13. [ 5x] Philip Johnson
 14. [ 4x] Elizabeth Weatherford
 15. [ 4x] Susana Torre
 16. [ 4x] Edith Thomas
 17. [ 4x] George Sand
 18. [ 4x] Gayle Rubin
 19. [ 4x] Rayna R. Reiter
 20. [ 4x] Ruth Crawford
 21. [ 4x] M. Rosaldo
 22. [ 4x] L. Lamphere
 23. [ 3x] Nancy Myron
 24. [ 3x] Louise Lamphere
 25. [ 3x] Emily Carr
 26. [ 3x] Amy Lowell
 27. [ 3x] Ann Sutherland
 28. [ 3x] H.W. Wilson
 29. [ 3x] Jan Oxenberg
 30. [ 3x] Barbara Grier
 31. [ 3x] Englewood Cliffs
 32. [ 3x] N. J.:
 33. [ 3x] Charles Rennie Mackintosh
 34. [ 3x] Kegan Paul
 35. [ 3x] John Wiley
 36. [ 3x] Van Nostrand

In [16]:
# Save full list as JSON
import json

names_data = [
    {
        'rank': i+1,
        'count': count,
        'name': name
    }
    for i, (name, count) in enumerate(sorted_names)
]

json_path = os.path.join(base_path, 'h2', 'h2_extracted_names.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(names_data, f, indent=2, ensure_ascii=False)

print(f"Saved {len(sorted_names)} names to h2_extracted_names.json")
print(f"Location: {json_path}")

Saved 1389 names to h2_extracted_names.json
Location: /Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed/h2/h2_extracted_names.json


In [17]:
#### 2.5.3 Complete Spelling Lookup Table

print("\n#### 2.5.3 Complete Spelling Lookup Table\n")

import json

# Map all identified variants to canonical forms
spelling_lookup = {
    # Ruth Crawford variants
    "Ruth Crawford": "Ruth Crawford Seeger",
    "Ruth Ctawiord-Seeger-": "Ruth Crawford Seeger",  # OCR error
    "Ruth Ctawford": "Ruth Crawford Seeger",  # OCR error
    
    # Rosaldo variants
    "M. Rosaldo": "Michelle Zimbalist Rosaldo",
    "Michelle Z. Rosaldo": "Michelle Zimbalist Rosaldo",
    
    # Rayna Reiter/Rapp variants (these may be the same person, research needed)
    "Rayna R. Reiter": "Rayna Reiter",
    "Rayna Rapp": "Rayna Reiter",  # Likely same person, married name change
    "R. Reiter": "Rayna Reiter",
    
    # Lamphere variants
    "L. Lamphere": "Louise Lamphere",
    
    # McAllester variants
    "D. McAllester": "David McAllester",
    
    # Lippard variants
    "Lucy R. Lippard": "Lucy Lippard",
    "Lucy R.": "Lucy Lippard",
    
    # Mead variants (OCR)
    "kita Mead": "Rita Mead",
    "Rita Head": "Rita Mead",  # Possible OCR error or variant
    
    # Sand variants
    "G. Sand": "George Sand",
    
    # Cowell variants (case)
    "Henry cowell": "Henry Cowell",
    
    # Additional variants observed in the full list
    "Vivian Gornick": "Vivian Gornick",  # Already canonical
    "V. Gornick": "Vivian Gornick",
    
    "Barbara K. Moran": "Barbara K. Moran",  # Already canonical
    "B K. Moran": "Barbara K. Moran",
    
    "Gerda Lerner": "Gerda Lerner",  # Already canonical
    
    # Susan Brownmiller
    "S. Brownmiller": "Susan Brownmiller",
    
    # Wanda Bacewicz (variant of Grazyna)
    "Wanda Bacewicz": "Wanda Bacewicz",  # Different person, keep separate
    
    # Clara Zetkin variants
    "Clara \"Zetkin": "Clara Zetkin",  # OCR error with quote mark
    
    # Rosa Luxemburg variants
    "Rosa Luremburg": "Rosa Luxemburg",  # OCR error
    
    # Ann Halprin variants
    "Ann Halprin's": "Ann Halprin",
    
    # Sherry Ortner variants
    "Sherry Ortners": "Sherry Ortner",
    
    # Joan Bamberger (appears once, keep as is)
    
    # Deirdre English variants (spelling consistency)
    "Deidre English": "Deirde English",
    "Diedre English": "Deirde English",
    
    # John Lomax variants
    "John Avery Lomax": "John Avery Lomax",  # Keep full name
    
    # Rita Mae variants (possibly incomplete)
    "Rita Mae": "Rita Mae Brown",  # Likely Rita Mae Brown
    
    # Eleanor Marx (appears once)
    
    # John Day (publisher, might remove from final)
    
    # De Colores (publication, might remove)
    
    # Susan Sontag (appears once)
    
    # Andrea Dworkin (appears once)
}

print(f"Spelling lookup table created with {len(spelling_lookup)} mappings:\n")
for raw, canonical in sorted(spelling_lookup.items()):
    if raw != canonical:
        print(f"  '{raw}' → '{canonical}'")

print(f"\n\nNote: Some mappings need verification:")
print(f"  - Rayna Reiter vs Rayna Rapp: Verify if this is the same person")
print(f"  - Rita Mae: Verify full name (likely Rita Mae Brown)")
print(f"  - Deirde/Diedre/Deidre: Verify correct spelling")

# Export lookup table for reference
lookup_path = os.path.join(base_path, 'h2', 'h2_spelling_lookup_table.json')
with open(lookup_path, 'w', encoding='utf-8') as f:
    json.dump(spelling_lookup, f, indent=2, ensure_ascii=False)
print(f"\n✓ Lookup table exported to: {lookup_path}")


#### 2.5.3 Complete Spelling Lookup Table

Spelling lookup table created with 31 mappings:

  'Ann Halprin's' → 'Ann Halprin'
  'B K. Moran' → 'Barbara K. Moran'
  'Clara "Zetkin' → 'Clara Zetkin'
  'D. McAllester' → 'David McAllester'
  'Deidre English' → 'Deirde English'
  'Diedre English' → 'Deirde English'
  'G. Sand' → 'George Sand'
  'Henry cowell' → 'Henry Cowell'
  'L. Lamphere' → 'Louise Lamphere'
  'Lucy R.' → 'Lucy Lippard'
  'Lucy R. Lippard' → 'Lucy Lippard'
  'M. Rosaldo' → 'Michelle Zimbalist Rosaldo'
  'Michelle Z. Rosaldo' → 'Michelle Zimbalist Rosaldo'
  'R. Reiter' → 'Rayna Reiter'
  'Rayna R. Reiter' → 'Rayna Reiter'
  'Rayna Rapp' → 'Rayna Reiter'
  'Rita Head' → 'Rita Mead'
  'Rita Mae' → 'Rita Mae Brown'
  'Rosa Luremburg' → 'Rosa Luxemburg'
  'Ruth Crawford' → 'Ruth Crawford Seeger'
  'Ruth Ctawford' → 'Ruth Crawford Seeger'
  'Ruth Ctawiord-Seeger-' → 'Ruth Crawford Seeger'
  'S. Brownmiller' → 'Susan Brownmiller'
  'Sherry Ortners' → 'Sherry Ortner'
  'V. Gor

#### 2.5.4 Normalize Reference List Names

In [18]:
#### 2.5.4 Normalize Reference List Names

print("\n#### 2.5.4 Normalize Reference List Names\n")

# Function to normalize names using lookup table
def normalize_name(raw_name, lookup_table):
    """
    Normalize a raw name using the lookup table.
    If not in lookup, return the raw name as-is.
    """
    return lookup_table.get(raw_name, raw_name)

# Apply normalization to all reference list entities
for entity in reference_list_entities:
    entity['canonical_name'] = normalize_name(entity['raw_name'], spelling_lookup)

# Statistics after normalization
unique_raw = len(set(e['raw_name'] for e in reference_list_entities))
unique_canonical = len(set(e['canonical_name'] for e in reference_list_entities))
reduction = unique_raw - unique_canonical

print(f"Normalization Statistics:")
print(f"  Total entities: {len(reference_list_entities)}")
print(f"  Unique raw names before: {unique_raw}")
print(f"  Unique canonical names after: {unique_canonical}")
print(f"  Reduction: {reduction} duplicate names merged")

# Calculate frequency per canonical name
canonical_totals = {}
for entity in reference_list_entities:
    canonical = entity['canonical_name']
    canonical_totals[canonical] = canonical_totals.get(canonical, 0) + 1

# Sort by frequency
sorted_canonical = sorted(canonical_totals.items(), key=lambda x: x[1], reverse=True)

print(f"\n\nTop 20 referenced thinkers in reference lists (after normalization):")
print("="*70)
for i, (name, count) in enumerate(sorted_canonical[:20]):
    print(f"  {i+1:2d}. [{count:3d}x] {name}")

print(f"\n\nSample of normalized entities (first 30):")
print("="*70)
for i, entity in enumerate(reference_list_entities[:30]):
    if entity['raw_name'] != entity['canonical_name']:
        print(f"  {i+1}. '{entity['raw_name']}' → '{entity['canonical_name']}'")
    else:
        print(f"  {i+1}. {entity['canonical_name']}")


#### 2.5.4 Normalize Reference List Names

Normalization Statistics:
  Total entities: 1656
  Unique raw names before: 1389
  Unique canonical names after: 1366
  Reduction: 23 duplicate names merged


Top 20 referenced thinkers in reference lists (after normalization):
   1. [ 13x] Ruth Crawford Seeger
   2. [ 10x] Rayna Reiter
   3. [  8x] Charles Seeger
   4. [  7x] Charlotte Bunch
   5. [  7x] Michelle Zimbalist Rosaldo
   6. [  7x] Louise Lamphere
   7. [  7x] Alan Lomax
   8. [  7x] Inez Garcia
   9. [  6x] Sheila Rowbotham
  10. [  6x] Lucy Lippard
  11. [  6x] Virginia Woolf
  12. [  6x] Romaine Brooks
  13. [  5x] Linda Nochlin
  14. [  5x] George Sand
  15. [  5x] Antonio Gramsci
  16. [  5x] Shirley Ardener
  17. [  5x] Grazyna Bacewicz
  18. [  5x] Rita Mead
  19. [  5x] Philip Johnson
  20. [  4x] Elizabeth Weatherford


Sample of normalized entities (first 30):
  1. Philip Schaff et al
  2. J. Offaolain
  3. L. Martines
  4. Ancrene Riwle
  5. Ancrene Riwle
  6. E. J. Do

#### 2.6 Paragraphs: NER → normalize → count variations

#### 2.6.1 NER on paragraph text

In [19]:
#### 2.6 Paragraphs: NER → Normalize → Count Variations

print("\n" + "="*80)
print("SECTION 2.6: NER on Paragraph Text")
print("="*80)

#### 2.6.1 NER on Paragraph Text

print("\n#### 2.6.1 NER on Paragraph Text\n")

# Extract PERSON entities from paragraphs (same filtering as reference lists)
paragraph_entities = []

# Use the same filter as before
filter_out = {
    # Single letter initials
    'j', 'i', 'k', 'c', 'p', 'r', 's', 'm', 'a', 'b', 'e', 'l', 'g', 'w', 'f', 'd', 'n',
    # Common first names only
    'baltimore', 'barbara', 'ann', 'ruth', 'margaret', 'simon', 'james', 'thomas', 
    'cal', 'dutton', 'linda', 'robert', 'morris', 'marshall', 'charles',
    # Common publishers, places, abbreviations
    'press', 'university', 'new', 'york', 'london', 'paris', 'cambridge', 'chicago',
    'harper', 'yale', 'oxford', 'princeton', 'mcgraw', 'hill', 'grove', 'ed', 'eds',
    'et', 'al', 'pp', 'vol', 'no', 'trans', 'intro', 'foreword', 'preface',
    'simon', 'schuster', 'penguin', 'doubleday', 'holt', 'rinehart', 'winston',
    # Specific publishers/journals
    'schocken books', 'william morrow', 'bantam books', 'sinister wisdom', 'lesbian tide',
    'w. w. norton', 'w.w. norton', 'norton',
    # Phrases misidentified as people
    'female militancy',
}

publisher_keywords = {'books', 'press', 'publications', 'publishers', 'journal', 'tide', 'wisdom', 'norton'}

for doc in h2_corpus_prepared:
    issue = doc['issue']
    year = doc['year']
    paragraph_text = doc['paragraph_text']
    
    # Process text with spaCy
    doc_processed = nlp(paragraph_text)
    
    # Extract PERSON entities
    for ent in doc_processed.ents:
        if ent.label_ == "PERSON":
            name = ent.text.strip()
            name_lower = name.lower()
            
            # Apply same filters as reference lists
            if name_lower in filter_out:
                continue
            
            if ' ' not in name:
                continue
            
            if len(name) < 5:
                continue
            
            if any(keyword in name_lower for keyword in publisher_keywords):
                continue
            
            if any(pattern in name_lower for pattern in 
                   ['press', 'university', 'pub', 'edition', 'vol', 'series']):
                continue
            
            # If it passes all filters, add it
            paragraph_entities.append({
                'source': 'paragraph',
                'raw_name': name,
                'issue': issue,
                'year': year
            })

print(f"Total PERSON entities extracted from paragraphs: {len(paragraph_entities)}")
print(f"\nSample extracted names (first 25):")
for i, entity in enumerate(paragraph_entities[:25]):
    print(f"  {i+1}. {entity['raw_name']}")


SECTION 2.6: NER on Paragraph Text

#### 2.6.1 NER on Paragraph Text

Total PERSON entities extracted from paragraphs: 5025

Sample extracted names (first 25):
  1. Patsy Beckert
  2. Joan Braderman
  3. Mary Beth Edelson
  4. Harmony Hammond
  5. Elizabeth Hess
  6. Joyce Kozloff
  7. Arlene Ladden
  8. Lucy Lippard
  9. Mary Miss
  10. Marty Pottenger
  11. Miriam Schapiro
  12. Joan Snyder
  13. Elke Solomon
  14. Pat Steir
  15. May Stevens
  16. Michelle Stuart
  17. Susana Torre
  18. Elizabeth Weatherford
  19. Sally Webster
  20. Nina Yankowitz
  21. O. Box
  22. OHeresies Collective
  23. Mandy Martin
  24. Myrna Zimmerman
  25. Harmony Hammond


#### 2.6.2 Normalize paragraph names

In [20]:
#### 2.6.2 Normalize Paragraph Names

print("\n#### 2.6.2 Normalize Paragraph Names\n")

# Apply the same normalization to paragraph entities
for entity in paragraph_entities:
    entity['canonical_name'] = normalize_name(entity['raw_name'], spelling_lookup)

# Statistics
unique_raw_para = len(set(e['raw_name'] for e in paragraph_entities))
unique_canonical_para = len(set(e['canonical_name'] for e in paragraph_entities))
reduction_para = unique_raw_para - unique_canonical_para

print(f"Paragraph Normalization Statistics:")
print(f"  Total entities: {len(paragraph_entities)}")
print(f"  Unique raw names before: {unique_raw_para}")
print(f"  Unique canonical names after: {unique_canonical_para}")
print(f"  Reduction: {reduction_para} duplicate names merged")

# Top canonical names in paragraphs
canonical_totals_para = {}
for entity in paragraph_entities:
    canonical = entity['canonical_name']
    canonical_totals_para[canonical] = canonical_totals_para.get(canonical, 0) + 1

sorted_canonical_para = sorted(canonical_totals_para.items(), key=lambda x: x[1], reverse=True)

print(f"\n\nTop 20 referenced thinkers in paragraphs (after normalization):")
print("="*70)
for i, (name, count) in enumerate(sorted_canonical_para[:20]):
    print(f"  {i+1:2d}. [{count:3d}x] {name}")


#### 2.6.2 Normalize Paragraph Names

Paragraph Normalization Statistics:
  Total entities: 5025
  Unique raw names before: 3447
  Unique canonical names after: 3445
  Reduction: 2 duplicate names merged


Top 20 referenced thinkers in paragraphs (after normalization):
   1. [ 29x] Harmony Hammond
   2. [ 24x] Sue Heinemann
   3. [ 23x] Su Friedrich
   4. [ 23x] Bessie Smith
   5. [ 22x] Lucy Lippard
   6. [ 20x] Miriam Schapiro
   7. [ 19x] Amy Sillman
   8. [ 18x] Elizabeth Weatherford
   9. [ 18x] Van Der Zee
  10. [ 17x] Joan Braderman
  11. [ 17x] Joyce Kozloff
  12. [ 16x] Arlene Ladden
  13. [ 16x] Marty Pottenger
  14. [ 16x] Susana Torre
  15. [ 16x] Virginia Woolf
  16. [ 16x] Janet Froelich
  17. [ 16x] Frida Kahlo
  18. [ 15x] Patsy Beckert
  19. [ 15x] Joan Snyder
  20. [ 15x] Pat Steir


#### 2.7 Comparison: Reference Lists vs Paragraph Mentions

In [21]:
#### 2.7 Comparison: Reference Lists vs Paragraph Mentions

print("\n" + "="*80)
print("SECTION 2.7: Comparison - Reference Lists vs Paragraph Mentions")
print("="*80 + "\n")

# Find which names appear in which sources
ref_list_names = set(e['canonical_name'] for e in reference_list_entities)
para_names = set(e['canonical_name'] for e in paragraph_entities)

both = ref_list_names & para_names
only_refs = ref_list_names - para_names
only_para = para_names - ref_list_names

print(f"Name Distribution Across Sources:")
print(f"  Appear in BOTH sources: {len(both)}")
print(f"  Appear ONLY in reference lists: {len(only_refs)}")
print(f"  Appear ONLY in paragraphs: {len(only_para)}")
print(f"  Total unique canonical names: {len(ref_list_names | para_names)}")

# Calculate combined frequency
print(f"\n\nTop 15 thinkers by COMBINED mentions (references + paragraphs):")
print("="*70)
combined_freq = {}
for name in ref_list_names | para_names:
    ref_count = canonical_totals.get(name, 0)
    para_count = canonical_totals_para.get(name, 0)
    combined_freq[name] = (ref_count + para_count, ref_count, para_count)

# Sort by total count
sorted_combined = sorted(combined_freq.items(), key=lambda x: x[1][0], reverse=True)

for i, (name, (total, ref_count, para_count)) in enumerate(sorted_combined[:15]):
    print(f"  {i+1:2d}. [{total:3d}x total] {name}")
    print(f"      └─ refs={ref_count:2d}x, para={para_count:2d}x")

# Names unique to each source
print(f"\n\nTop 10 names ONLY in reference lists:")
print("="*70)
only_refs_sorted = sorted([(name, canonical_totals.get(name, 0)) for name in only_refs], 
                          key=lambda x: x[1], reverse=True)
for i, (name, count) in enumerate(only_refs_sorted[:10]):
    print(f"  {i+1}. [{count:2d}x] {name}")

print(f"\n\nTop 10 names ONLY in paragraphs:")
print("="*70)
only_para_sorted = sorted([(name, canonical_totals_para.get(name, 0)) for name in only_para], 
                          key=lambda x: x[1], reverse=True)
for i, (name, count) in enumerate(only_para_sorted[:10]):
    print(f"  {i+1}. [{count:2d}x] {name}")


SECTION 2.7: Comparison - Reference Lists vs Paragraph Mentions

Name Distribution Across Sources:
  Appear in BOTH sources: 277
  Appear ONLY in reference lists: 1089
  Appear ONLY in paragraphs: 3168
  Total unique canonical names: 4534


Top 15 thinkers by COMBINED mentions (references + paragraphs):
   1. [ 31x total] Harmony Hammond
      └─ refs= 2x, para=29x
   2. [ 28x total] Lucy Lippard
      └─ refs= 6x, para=22x
   3. [ 25x total] Ruth Crawford Seeger
      └─ refs=13x, para=12x
   4. [ 24x total] Sue Heinemann
      └─ refs= 0x, para=24x
   5. [ 24x total] Bessie Smith
      └─ refs= 1x, para=23x
   6. [ 24x total] Su Friedrich
      └─ refs= 1x, para=23x
   7. [ 22x total] Miriam Schapiro
      └─ refs= 2x, para=20x
   8. [ 22x total] Virginia Woolf
      └─ refs= 6x, para=16x
   9. [ 22x total] Elizabeth Weatherford
      └─ refs= 4x, para=18x
  10. [ 20x total] Susana Torre
      └─ refs= 4x, para=16x
  11. [ 19x total] Amy Sillman
      └─ refs= 0x, para=19x
  12. [ 1

#### 2.8 Output Construction

- Produce structured reference frequency table
- Columns: (canonical_name, raw_variant, issue, year, count)
- Export as corpus_h2_references.csv
- Summary: top referenced thinkers overall and per period

In [22]:
#### 2.8 Output Construction

print("\n" + "="*80)
print("SECTION 2.8: Output Construction")
print("="*80 + "\n")

# Combine all entities (reference lists + paragraphs)
all_entities = reference_list_entities + paragraph_entities

# Create output records: (canonical_name, raw_variant, source, issue, year, count)
output_records = []

# Group by (canonical_name, raw_variant, source, issue, year) and count
from collections import defaultdict
combinations = defaultdict(int)

for entity in all_entities:
    key = (entity['canonical_name'], entity['raw_name'], entity['source'], entity['issue'], entity['year'])
    combinations[key] += 1

# Convert to records
for (canonical_name, raw_variant, source, issue, year), count in combinations.items():
    output_records.append({
        'canonical_name': canonical_name,
        'raw_variant': raw_variant,
        'source': source,
        'issue': issue,
        'year': year,
        'count': count
    })

# Create DataFrame
df_h2_references = pd.DataFrame(output_records)
df_h2_references = df_h2_references.sort_values('count', ascending=False).reset_index(drop=True)

print(f"Output DataFrame Summary:")
print(f"  Total records: {len(df_h2_references)}")
print(f"  Unique canonical names: {df_h2_references['canonical_name'].nunique()}")
print(f"  Unique raw variants: {df_h2_references['raw_variant'].nunique()}")
print(f"  Issues covered: {df_h2_references['issue'].nunique()}")
print(f"  Years covered: {sorted(df_h2_references['year'].unique())}")
print(f"  Sources: {df_h2_references['source'].unique().tolist()}")

print(f"\n\nTop 25 referenced thinkers (all sources combined):")
print("="*70)
top_by_canonical = df_h2_references.groupby('canonical_name')['count'].sum().sort_values(ascending=False)
for i, (name, count) in enumerate(top_by_canonical.head(25).items()):
    print(f"  {i+1:2d}. [{count:3d}x] {name}")

# Export to CSV
output_path = os.path.join(base_path, 'h2', 'corpus_h2_references.csv')
df_h2_references.to_csv(output_path, index=False)
print(f"\n✓ Exported: {output_path}")

print(f"\n\nFirst 30 rows of output file:")
print("="*70)
print(df_h2_references.head(30).to_string(index=False))

# Summary: Top referenced thinkers per year
print(f"\n\n" + "="*80)
print("Summary: Top 5 Referenced Thinkers Per Year")
print("="*80)

for year in sorted(df_h2_references['year'].dropna().unique()):
    year = int(year)
    year_data = df_h2_references[df_h2_references['year'] == year]
    top_year = year_data.groupby('canonical_name')['count'].sum().sort_values(ascending=False).head(5)
    
    print(f"\n{year}:")
    for rank, (name, count) in enumerate(top_year.items(), 1):
        print(f"  {rank}. [{count:2d}x] {name}")


SECTION 2.8: Output Construction

Output DataFrame Summary:
  Total records: 5512
  Unique canonical names: 4534
  Unique raw variants: 4559
  Issues covered: 11
  Years covered: [np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981)]
  Sources: ['paragraph', 'reference_list']


Top 25 referenced thinkers (all sources combined):
   1. [ 31x] Harmony Hammond
   2. [ 28x] Lucy Lippard
   3. [ 25x] Ruth Crawford Seeger
   4. [ 24x] Sue Heinemann
   5. [ 24x] Su Friedrich
   6. [ 24x] Bessie Smith
   7. [ 22x] Elizabeth Weatherford
   8. [ 22x] Virginia Woolf
   9. [ 22x] Miriam Schapiro
  10. [ 20x] Susana Torre
  11. [ 19x] Amy Sillman
  12. [ 19x] Joyce Kozloff
  13. [ 19x] Charles Seeger
  14. [ 18x] Van Der Zee
  15. [ 17x] Joan Braderman
  16. [ 16x] George Sand
  17. [ 16x] Arlene Ladden
  18. [ 16x] Frida Kahlo
  19. [ 16x] Pat Steir
  20. [ 16x] Marty Pottenger
  21. [ 16x] Janet Froelich
  22. [ 15x] Elizabeth Hess
  23. [ 15x] Joan Snyder
  24. [ 15x] M

#### Summary: Top referenced thinkers per year

In [29]:
print("="*70)
print("TOP REFERENCED THINKERS BY VOLUME")
print("="*70)

# Function to get volume from issue number
def get_volume(issue):
    issue_num = int(issue)
    return (issue_num - 1) // 4 + 1

# Add volume column
df_h2_references['volume'] = df_h2_references['issue'].astype(int).apply(get_volume)

volume_summary_records = []

for volume in sorted(df_h2_references['volume'].unique()):
    volume_data = df_h2_references[df_h2_references['volume'] == volume]
    
    print(f"\nVolume {volume}:")
    print(f"  Reference Lists:")
    ref_data = volume_data[volume_data['source'] == 'reference_list'].groupby('canonical_name')['count'].sum().sort_values(ascending=False).head(5)
    for rank, (name, count) in enumerate(ref_data.items(), 1):
        print(f"    {rank}. [{count:3d}x] {name}")
        volume_summary_records.append({'volume': volume, 'source': 'reference_list', 'rank': rank, 'canonical_name': name, 'count': count})
    
    print(f"  Paragraphs:")
    para_data = volume_data[volume_data['source'] == 'paragraph'].groupby('canonical_name')['count'].sum().sort_values(ascending=False).head(5)
    for rank, (name, count) in enumerate(para_data.items(), 1):
        print(f"    {rank}. [{count:3d}x] {name}")
        volume_summary_records.append({'volume': volume, 'source': 'paragraph', 'rank': rank, 'canonical_name': name, 'count': count})

# Export volume summary to CSV
df_volume_summary = pd.DataFrame(volume_summary_records)
summary_path = os.path.join(base_path, 'h2', 'corpus_h2_references_volume_summary.csv')
df_volume_summary.to_csv(summary_path, index=False)
print(f"\n✓ Exported volume summary: {summary_path}")

TOP REFERENCED THINKERS BY VOLUME

Volume 1:
  Reference Lists:
    1. [  7x] Charlotte Bunch
    2. [  6x] Romaine Brooks
    3. [  5x] Sheila Rowbotham
    4. [  5x] Antonio Gramsci
    5. [  5x] George Sand
  Paragraphs:
    1. [ 16x] Frida Kahlo
    2. [ 14x] Harmony Hammond
    3. [ 11x] Su Friedrich
    4. [ 11x] George Sand
    5. [ 10x] Miriam Schapiro

Volume 2:
  Reference Lists:
    1. [  7x] Inez Garcia
    2. [  4x] Rayna Reiter
    3. [  3x] Fran P. Hosken
    4. [  2x] Robert Murphy
    5. [  2x] Gayle Rubin
  Paragraphs:
    1. [ 18x] Van Der Zee
    2. [  6x] Sue Heinemann
    3. [  6x] Su Friedrich
    4. [  5x] Patsy Beckert
    5. [  5x] Jane Silver

Volume 3:
  Reference Lists:
    1. [ 13x] Ruth Crawford Seeger
    2. [  8x] Charles Seeger
    3. [  5x] Rita Mead
    4. [  5x] Grazyna Bacewicz
    5. [  5x] Alan Lomax
  Paragraphs:
    1. [ 21x] Bessie Smith
    2. [ 12x] Sue Heinemann
    3. [ 12x] Julia Morgan
    4. [ 12x] Ruth Crawford Seeger
    5. [ 11x] Cha

## 3. Hypothesis 3 - Contributor Metadata Dataset Construction

**Hypothesis 3: Contributors and Collaboration**

Shifts in feminist theoretical orientation within Heresies will coincide with changes in patterns of contribution, including the emergence of new contributors and changing thematic emphases associated with the magazine’s rotating collectives.

#### 3.1 H3: Contributor Dataset Construction

#### 3.1.1 Extract and Parse Author/Creator and Caption Fields

In [36]:
print("\n" + "="*80)
print("SECTION 3.4: NER on Contributor Data (Author/Creator and Captions)")
print("="*80)

import re
import pandas as pd

# Load H3 corpus from CSV
h3_df = pd.read_csv(os.path.join(base_path, 'h3', 'corpus_h3.csv'))

print(f"H3 Corpus loaded: {len(h3_df)} issues")

print("\n#### 3.4.1 Extract and Parse Author/Creator and Caption Fields\n")

h3_entities = []

for idx, row in h3_df.iterrows():
    issue = row['issue']
    year = row['year']
    text = row['text']
    
    # Extract author_creator tags and apply NER
    author_creators = re.findall(r'<author_creator>\s*(.*?)\s*</author_creator>', text, re.DOTALL)
    for author in author_creators:
        author = author.strip()
        if author and len(author) > 2:
            # Apply NER to author_creator text to extract person names
            doc_processed = nlp(author)
            for ent in doc_processed.ents:
                if ent.label_ == "PERSON":
                    name = ent.text.strip()
                    # Basic filters
                    if ' ' in name and len(name) > 4:
                        h3_entities.append({
                            'source': 'author_creator',
                            'raw_name': name,
                            'issue': issue,
                            'year': year
                        })
    
    # Extract caption tags and apply NER
    captions = re.findall(r'<caption>\s*(.*?)\s*</caption>', text, re.DOTALL)
    for caption in captions:
        caption = caption.strip()
        if caption and len(caption) > 2:
            # Extract person names from caption using spaCy
            doc_processed = nlp(caption)
            for ent in doc_processed.ents:
                if ent.label_ == "PERSON":
                    name = ent.text.strip()
                    # Basic filters
                    if ' ' in name and len(name) > 4:
                        h3_entities.append({
                            'source': 'caption',
                            'raw_name': name,
                            'issue': issue,
                            'year': year
                        })

print(f"Total entities extracted from author_creator and captions: {len(h3_entities)}")
print(f"\nSample extracted names (first 25):")
for i, entity in enumerate(h3_entities[:25]):
    print(f"  {i+1}. [{entity['source']}] {entity['raw_name']}")


SECTION 3.4: NER on Contributor Data (Author/Creator and Captions)
H3 Corpus loaded: 11 issues

#### 3.4.1 Extract and Parse Author/Creator and Caption Fields

Total entities extracted from author_creator and captions: 1027

Sample extracted names (first 25):
  1. [author_creator] Barbara Ehrenreich
  2. [author_creator] Mortha Rosler
  3. [author_creator] Adrienne Rich
  4. [author_creator] Elizabeth Zelvin
  5. [author_creator] Carol Muske
  6. [author_creator] Carole Ramer
  7. [author_creator] Deborah Hiller
  8. [author_creator] Gloria Jensen
  9. [author_creator] Arlene Ladden
  10. [author_creator] Carol Duncan
  11. [author_creator] Susan Yankowitz
  12. [author_creator] Jayne Cortez
  13. [author_creator] Jon Clausen
  14. [author_creator] Meridel LeSueur
  15. [author_creator] Susan Saxe
  16. [author_creator] Harmony Hommond
  17. [author_creator] Ruth E. Iskin
  18. [author_creator] Lucy R. Lippard
  19. [author_creator] Joan Braderman
  20. [author_creator] Kate Jennings


#### 3.1.2 Identify Spelling variations

In [41]:
from collections import defaultdict
import json

# Count raw name frequencies
raw_name_freq = defaultdict(int)
for entity in h3_entities:
    raw_name_freq[entity['raw_name']] += 1

# Display top names
sorted_names = sorted(raw_name_freq.items(), key=lambda x: x[1], reverse=True)
print(f"Top 50 most frequently extracted contributors:")
print("="*70)
for i, (name, count) in enumerate(sorted_names[:50]):
    print(f"{i+1:2d}. [{count:3d}x] {name}")

# Save full list as JSON
names_data = [
    {
        'rank': i+1,
        'count': count,
        'name': name
    }
    for i, (name, count) in enumerate(sorted_names)
]

json_path = os.path.join(base_path, 'h3', 'h3_extracted_names.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(names_data, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved {len(sorted_names)} names to h3_extracted_names.json")
print(f"  Location: {json_path}")

Top 50 most frequently extracted contributors:
 1. [ 11x] Su Friedrich
 2. [ 10x] JOAN BRADERMAN
 3. [  6x] Amy Sillman
 4. [  6x] Alice Austen
 5. [  5x] Frida Kahlo
 6. [  5x] Howardena Pindell
 7. [  5x] Roberta Neiman
 8. [  5x] Nancy Fried
 9. [  5x] Dee Shapiro
10. [  5x] V. E. Browne
11. [  5x] Gwendolyn Wright
12. [  4x] Louise Fishman
13. [  4x] Harmony Hammond
14. [  4x] Judy Silberstein
15. [  4x] Valerie Harris
16. [  4x] Marge Dean
17. [  3x] Barbara Ehrenreich
18. [  3x] Adrienne Rich
19. [  3x] Susan Saxe
20. [  3x] Lucy R. Lippard
21. [  3x] Marty Pottenger
22. [  3x] Joan Semmel
23. [  3x] Mary Beth Edelson
24. [  3x] Miriam Schapiro
25. [  3x] D. James Dee
26. [  3x] Cynthia Carr
27. [  3x] Joan Nestle
28. [  3x] Carol Bloom
29. [  3x] Fran Winant
30. [  3x] Lynda Benglis
31. [  3x] Barbara Zucker
32. [  3x] Sarah Mandell
33. [  3x] Batya Weinbaum
34. [  3x] Leslie Labowitz
35. [  3x] Gail Lineback
36. [  3x] Paula Webster
37. [  3x] Janet Goldner
38. [  3x] Betye Saa

#### 3.4.3 Review and Build Manual Spelling Lookup Table

In [43]:
#### 3.4.3 Build Manual Spelling Lookup Table

# Create spelling lookup table for H3 with identified variants
h3_spelling_lookup = {
    # Capitalization variants
    "JOAN BRADERMAN": "Joan Braderman",
    "SONDRA SEGAL": "Sondra Segal",
    "PAT CALIFIA": "Pat Califia",
    "MARY WINSTON": "Mary Winston",
    "J. LEE LEHMAN": "J. Lee Lehman",
    
    # Abbreviated/full name variants
    "V. E. Browne": "V. E. Browne",  # Keep as is - abbreviated form is how credited
    "D. James Dee": "D. James Dee",  # Keep as is
    "Lucy R. Lippard": "Lucy R. Lippard",  # Keep as is
    "M. W. Duncan": "M. W. Duncan",  # Keep as is
    "C.F. Peters": "C.F. Peters",  # Keep as is
    "A. Broude": "A. Broude",  # Keep as is
    
    # Lowercase variants (likely OCR or transcription errors)
    "hattie gossett": "Hattie Gossett",
    
    # Name variations / possible typos
    "Nunzia Rondamini": "Nunzia Rondanini",  # Spelling variant
    "Katrin Adam": "Katrin Aadam",  # Spelling variant
    "Susan Aitcheson": "Susan E. Aitcheson",  # Abbreviated vs full
    "Sandi Feliman": "Sandi Fellman",  # Spelling variant (check if appears elsewhere)
    "Janet Ruth Heuer": "Janet Ruth Heller",  # Possible OCR error or name variant
    "Micke McGee": "Micki McGee",  # Spelling variant
    "Seph Weene": "Seph Weene",  # Keep as is
    "Jane MGroarty": "Jane McGarty",  # OCR error - "M" should be space
    
    # Possessive forms (remove possessive)
    "Julia Morgan's": "Julia Morgan",
    "Ann Halprin's": "Ann Halprin",  # if appears
    
    # Full names where abbreviated forms also exist
    "J. Lee Lehman": "J. Lee Lehman",  # Canonical form
    "Janet Ruth Heller": "Janet Ruth Heller",  # Canonical form
    
    # OCR/Caption noise to filter or consolidate
    # Note: Some entries like "Lesbian wall", "Main Gate", "Manuscript Library", etc. 
    # appear to be OCR errors or non-person entities extracted by NER
    # These should be reviewed manually, but example mappings:
    "Lesbian wall": None,  # Mark for exclusion
    "Simply Obtained": None,  # Artwork title, not person
    "Eros Kalos": None,  # Artwork title
}

# Remove None entries (items to exclude)
h3_spelling_lookup = {k: v for k, v in h3_spelling_lookup.items() if v is not None}

print(f"Spelling lookup table created with {len(h3_spelling_lookup)} mappings:\n")
for raw, canonical in sorted(h3_spelling_lookup.items()):
    if raw != canonical:
        print(f"  '{raw}' → '{canonical}'")

print(f"\n\nNote: Many contributors appear with clean, consistent names.")
print(f"      Most capitalization differences are due to formatting in source data.")
print(f"      Some entries (particularly at rank 700+) are OCR artifacts or titles.")

Spelling lookup table created with 24 mappings:

  'Ann Halprin's' → 'Ann Halprin'
  'J. LEE LEHMAN' → 'J. Lee Lehman'
  'JOAN BRADERMAN' → 'Joan Braderman'
  'Jane MGroarty' → 'Jane McGarty'
  'Janet Ruth Heuer' → 'Janet Ruth Heller'
  'Julia Morgan's' → 'Julia Morgan'
  'Katrin Adam' → 'Katrin Aadam'
  'MARY WINSTON' → 'Mary Winston'
  'Micke McGee' → 'Micki McGee'
  'Nunzia Rondamini' → 'Nunzia Rondanini'
  'PAT CALIFIA' → 'Pat Califia'
  'SONDRA SEGAL' → 'Sondra Segal'
  'Sandi Feliman' → 'Sandi Fellman'
  'Susan Aitcheson' → 'Susan E. Aitcheson'
  'hattie gossett' → 'Hattie Gossett'


Note: Many contributors appear with clean, consistent names.
      Most capitalization differences are due to formatting in source data.
      Some entries (particularly at rank 700+) are OCR artifacts or titles.


#### 3.1.4 Normalize Contributor Names

In [52]:
#### 3.4.4 Normalize Contributor Names

print("\n#### 3.4.4 Normalize Contributor Names\n")

# Function to normalize names
def normalize_name(raw_name, lookup_table):
    return lookup_table.get(raw_name, raw_name)

# Apply normalization
for entity in h3_entities:
    entity['canonical_name'] = normalize_name(entity['raw_name'], h3_spelling_lookup)

# Statistics
unique_raw = len(set(e['raw_name'] for e in h3_entities))
unique_canonical = len(set(e['canonical_name'] for e in h3_entities))

print(f"Normalization Statistics:")
print(f"  Total entities: {len(h3_entities)}")
print(f"  Unique raw names before: {unique_raw}")
print(f"  Unique canonical names after: {unique_canonical}")

# Top contributors
canonical_totals = {}
for entity in h3_entities:
    canonical = entity['canonical_name']
    canonical_totals[canonical] = canonical_totals.get(canonical, 0) + 1

sorted_canonical = sorted(canonical_totals.items(), key=lambda x: x[1], reverse=True)

print(f"\n\nTop 30 contributors (after normalization):")
print("="*70)
for i, (name, count) in enumerate(sorted_canonical[:30]):
    print(f"  {i+1:2d}. [{count:3d}x] {name}")


#### 3.4.4 Normalize Contributor Names

Normalization Statistics:
  Total entities: 1027
  Unique raw names before: 767
  Unique canonical names after: 755


Top 30 contributors (after normalization):
   1. [ 11x] Joan Braderman
   2. [ 11x] Su Friedrich
   3. [  6x] Amy Sillman
   4. [  6x] Alice Austen
   5. [  5x] Frida Kahlo
   6. [  5x] Howardena Pindell
   7. [  5x] Roberta Neiman
   8. [  5x] Nancy Fried
   9. [  5x] Dee Shapiro
  10. [  5x] V. E. Browne
  11. [  5x] Gwendolyn Wright
  12. [  4x] Louise Fishman
  13. [  4x] Harmony Hammond
  14. [  4x] Judy Silberstein
  15. [  4x] Valerie Harris
  16. [  4x] Marge Dean
  17. [  3x] Barbara Ehrenreich
  18. [  3x] Adrienne Rich
  19. [  3x] Susan Saxe
  20. [  3x] Lucy R. Lippard
  21. [  3x] Marty Pottenger
  22. [  3x] Joan Semmel
  23. [  3x] Mary Beth Edelson
  24. [  3x] Miriam Schapiro
  25. [  3x] D. James Dee
  26. [  3x] Cynthia Carr
  27. [  3x] Joan Nestle
  28. [  3x] Carol Bloom
  29. [  3x] Fran Winant
  30. [  3x

#### 3.6 Linkage to Issue and Year Metadata

In [53]:
print("\n#### 3.4.6 Linkage to Issue and Year Metadata\n")

# Verify issue and year data
print("Checking issue and year metadata:")
print(f"  Total records: {len(df_h3_contributors)}")
print(f"  Records with missing year: {df_h3_contributors['year'].isna().sum()}")
print(f"  Records with missing issue: {df_h3_contributors['issue'].isna().sum()}")

# Show issues by year
print("\nIssues per year:")
issues_per_year = df_h3_contributors.groupby('year')['issue'].nunique()
for year in sorted(issues_per_year.index):
    print(f"  {year}: {issues_per_year[year]} issues")

# Verify all years are present
if df_h3_contributors['year'].isna().sum() > 0:
    print("\n⚠️  WARNING: Some records have missing year data. Review manually.")
else:
    print("\n✓ All records have valid year metadata.")

# Create year mapping for reference
year_issue_map = df_h3_contributors[['issue', 'year']].drop_duplicates().sort_values('issue')
print(f"\nIssue-Year Mapping:")
print(year_issue_map.to_string(index=False))


#### 3.4.6 Linkage to Issue and Year Metadata

Checking issue and year metadata:
  Total records: 900
  Records with missing year: 0
  Records with missing issue: 0

Issues per year:
  1977: 3 issues
  1978: 2 issues
  1979: 2 issues
  1980: 2 issues
  1981: 2 issues

✓ All records have valid year metadata.

Issue-Year Mapping:
 issue  year
     1  1977
     2  1977
     3  1977
     4  1978
     6  1978
     7  1979
     8  1979
     9  1980
    10  1980
    11  1981
    12  1981


#### 3.7 Output Construction

In [54]:
print("\n#### 3.4.5 Output Construction\n")

# Create output records: (canonical_name, raw_variant, source, issue, year, count)
output_records = []

# Group by (canonical_name, raw_variant, source, issue, year) and count
from collections import defaultdict
combinations = defaultdict(int)

for entity in h3_entities:
    key = (entity['canonical_name'], entity['raw_name'], entity['source'], entity['issue'], entity['year'])
    combinations[key] += 1

# Convert to records
for (canonical_name, raw_variant, source, issue, year), count in combinations.items():
    output_records.append({
        'canonical_name': canonical_name,
        'raw_variant': raw_variant,
        'source': source,
        'issue': issue,
        'year': year,
        'count': count
    })

# Create DataFrame
df_h3_contributors = pd.DataFrame(output_records)
df_h3_contributors = df_h3_contributors.sort_values('count', ascending=False).reset_index(drop=True)

print(f"Output DataFrame Summary:")
print(f"  Total records: {len(df_h3_contributors)}")
print(f"  Unique canonical names: {df_h3_contributors['canonical_name'].nunique()}")
print(f"  Unique raw variants: {df_h3_contributors['raw_variant'].nunique()}")
print(f"  Issues covered: {df_h3_contributors['issue'].nunique()}")
print(f"  Years covered: {sorted(df_h3_contributors['year'].unique())}")
print(f"  Sources: {df_h3_contributors['source'].unique().tolist()}")

print(f"\n\nTop 25 contributors (all sources combined):")
print("="*70)
top_by_canonical = df_h3_contributors.groupby('canonical_name')['count'].sum().sort_values(ascending=False)
for i, (name, count) in enumerate(top_by_canonical.head(25).items()):
    print(f"  {i+1:2d}. [{count:3d}x] {name}")

# Export to CSV
output_path = os.path.join(base_path, 'h3', 'corpus_h3_contributors.csv')
df_h3_contributors.to_csv(output_path, index=False)
print(f"\n✓ Exported: {output_path}")

print(f"\n\nFirst 30 rows of output file:")
print("="*70)
print(df_h3_contributors.head(30).to_string(index=False))


#### 3.4.5 Output Construction

Output DataFrame Summary:
  Total records: 900
  Unique canonical names: 755
  Unique raw variants: 767
  Issues covered: 11
  Years covered: [np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981)]
  Sources: ['caption', 'author_creator']


Top 25 contributors (all sources combined):
   1. [ 11x] Su Friedrich
   2. [ 11x] Joan Braderman
   3. [  6x] Amy Sillman
   4. [  6x] Alice Austen
   5. [  5x] Nancy Fried
   6. [  5x] Howardena Pindell
   7. [  5x] V. E. Browne
   8. [  5x] Frida Kahlo
   9. [  5x] Dee Shapiro
  10. [  5x] Roberta Neiman
  11. [  5x] Gwendolyn Wright
  12. [  4x] Judy Silberstein
  13. [  4x] Valerie Harris
  14. [  4x] Louise Fishman
  15. [  4x] Harmony Hammond
  16. [  4x] Marge Dean
  17. [  3x] Barbara DeGenevieve
  18. [  3x] Barbara Ehrenreich
  19. [  3x] Batya Weinbaum
  20. [  3x] Barbara Marks
  21. [  3x] Gail Lineback
  22. [  3x] Hattie Gossett
  23. [  3x] Barbara Zucker
  24. [  3x] Miriam 

#### Top Contributors by Volume

In [55]:
print("\n" + "="*70)
print("TOP CONTRIBUTORS BY VOLUME")
print("="*70)

# Function to get volume from issue number
def get_volume(issue):
    issue_num = int(issue)
    return (issue_num - 1) // 4 + 1

# Add volume column
df_h3_contributors['volume'] = df_h3_contributors['issue'].astype(int).apply(get_volume)

volume_summary_records = []

for volume in sorted(df_h3_contributors['volume'].unique()):
    volume_data = df_h3_contributors[df_h3_contributors['volume'] == volume]
    
    print(f"\nVolume {volume}:")
    print(f"  Author/Creator (Top 5):")
    author_data = volume_data[volume_data['source'] == 'author_creator'].groupby('canonical_name')['count'].sum().sort_values(ascending=False).head(5)
    for rank, (name, count) in enumerate(author_data.items(), 1):
        print(f"    {rank}. [{count:3d}x] {name}")
        volume_summary_records.append({'volume': volume, 'source': 'author_creator', 'rank': rank, 'canonical_name': name, 'count': count})
    
    print(f"  Caption (Top 5):")
    caption_data = volume_data[volume_data['source'] == 'caption'].groupby('canonical_name')['count'].sum().sort_values(ascending=False).head(5)
    for rank, (name, count) in enumerate(caption_data.items(), 1):
        print(f"    {rank}. [{count:3d}x] {name}")
        volume_summary_records.append({'volume': volume, 'source': 'caption', 'rank': rank, 'canonical_name': name, 'count': count})

# Export volume summary to CSV
df_volume_summary = pd.DataFrame(volume_summary_records)
summary_path = os.path.join(base_path, 'h3', 'corpus_h3_contributors_volume_summary.csv')
df_volume_summary.to_csv(summary_path, index=False)
print(f"\n✓ Exported volume summary: {summary_path}")


TOP CONTRIBUTORS BY VOLUME

Volume 1:
  Author/Creator (Top 5):
    1. [  5x] Joan Braderman
    2. [  3x] Amy Sillman
    3. [  2x] Dee Shapiro
    4. [  2x] Arlene Ladden
    5. [  2x] Cynthia Carr
  Caption (Top 5):
    1. [  6x] Alice Austen
    2. [  6x] Su Friedrich
    3. [  5x] Frida Kahlo
    4. [  5x] Roberta Neiman
    5. [  3x] Sarah Mandell

Volume 2:
  Author/Creator (Top 5):
    1. [  4x] Valerie Harris
    2. [  2x] Joan Braderman
    3. [  2x] Leslie Labowitz
    4. [  1x] Linda Lombardo
    5. [  1x] Maryann King
  Caption (Top 5):
    1. [  5x] V. E. Browne
    2. [  3x] Janet Goldner
    3. [  3x] Gail Lineback
    4. [  2x] Lula Blocton
    5. [  2x] Terry Wolverton

Volume 3:
  Author/Creator (Top 5):
    1. [  4x] Joan Braderman
    2. [  3x] Hattie Gossett
    3. [  2x] Sandra M. Whisler
    4. [  2x] Su Friedrich
    5. [  2x] Harmony Hammond
  Caption (Top 5):
    1. [  4x] Gwendolyn Wright
    2. [  4x] Marge Dean
    3. [  3x] Doranne Jacobson
    4. [  3x]

# 6. Reproducibility and Workflow Summary

#### 6.1 Output Inventory
- List all exported files with paths, descriptions, and row/token counts
- Confirm all three hypothesis datasets are present and complete

In [50]:
print("="*80)
print("OUTPUT INVENTORY")
print("="*80)

import os
from pathlib import Path

# Define output files
output_files = [
    {
        'file': 'h1_frequency_per_volume.csv',
        'path': os.path.join(base_path, 'h1', 'h1_frequency_per_volume.csv'),
        'description': 'H1: Frequency of tokens per volume',
        'hypothesis': 'Hypothesis 1 (Conceptual Shifts)'
    },
    {
        'file': 'corpus_h2_references.csv',
        'path': os.path.join(base_path, 'h2', 'corpus_h2_references.csv'),
        'description': 'H2: Person names extracted from reference lists and paragraph text',
        'hypothesis': 'Hypothesis 2 (References and Influences)'
    },
    {
        'file': 'corpus_h2_references_volume_summary.csv',
        'path': os.path.join(base_path, 'h2', 'corpus_h2_references_volume_summary.csv'),
        'description': 'H2: Top referenced thinkers per volume, separated by source',
        'hypothesis': 'Hypothesis 2 (References and Influences)'
    },
    {
        'file': 'corpus_h3_contributors.csv',
        'path': os.path.join(base_path, 'h3', 'corpus_h3_contributors.csv'),
        'description': 'H3: Contributors extracted from author/creator credits and captions',
        'hypothesis': 'Hypothesis 3 (Contributors and Collaboration)'
    },
    {
        'file': 'corpus_h3_contributors_volume_summary.csv',
        'path': os.path.join(base_path, 'h3', 'corpus_h3_contributors_volume_summary.csv'),
        'description': 'H3: Top contributors per volume, separated by source',
        'hypothesis': 'Hypothesis 3 (Contributors and Collaboration)'
    },
]

# Check which files exist and report
print("\nFile Status:\n")
all_present = True
for item in output_files:
    exists = os.path.exists(item['path'])
    status = "✓" if exists else "✗"
    print(f"{status} {item['file']}")
    print(f"   Path: {item['path']}")
    print(f"   Description: {item['description']}")
    print(f"   Hypothesis: {item['hypothesis']}")
    
    if exists:
        try:
            df = pd.read_csv(item['path'])
            print(f"   Records: {len(df):,}")
            print(f"   Columns: {', '.join(df.columns)}")
        except Exception as e:
            print(f"   Error reading file: {e}")
    else:
        all_present = False
    print()

# Summary
print("="*80)
if all_present:
    print("✓ All output files present and accounted for.")
else:
    print("⚠️  Some output files are missing. Check paths above.")
print("="*80)

OUTPUT INVENTORY

File Status:

✓ h1_frequency_per_volume.csv
   Path: /Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed/h1/h1_frequency_per_volume.csv
   Description: H1: Frequency of tokens per volume
   Hypothesis: Hypothesis 1 (Conceptual Shifts)
   Records: 60,740
   Columns: volume, token, frequency

✓ corpus_h2_references.csv
   Path: /Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed/h2/corpus_h2_references.csv
   Description: H2: Person names extracted from reference lists and paragraph text
   Hypothesis: Hypothesis 2 (References and Influences)
   Records: 5,512
   Columns: canonical_name, raw_variant, source, issue, year, count

✓ corpus_h2_references_volume_summary.csv
   Path: /Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed/h2/corpus_h2_references_volume_summary.csv
   Description: H2: Top referenced thinkers per volume, separated by source
   

#### 6.2 Cleaning Decision Log
- Markdown cell summarizing every cleaning decision made in this notebook, with the justification in one sentence each
- This can be copy-pasted directly into your thesis methods section

In [51]:
print("\n" + "="*80)
print("CLEANING DECISION LOG")
print("="*80 + "\n")

cleaning_decisions = """
### Hypothesis 1: Lexical Analysis

1. **Tokenization Method**: Used NLTK's word_tokenize() to segment text into word-level tokens. 
   Justification: Preserves hyphenated compounds and contractions for domain-appropriate analysis.

2. **Lowercasing**: Applied consistent lowercasing to all tokens before lemmatization. 
   Justification: Necessary for reliable type frequency comparison without introducing conceptual difference.

3. **Lemmatization**: Applied WordNetLemmatizer to reduce inflected forms to base lemmas. 
   Justification: Groups grammatical variants for more stable frequency analysis of key concepts.

4. **Stopword Removal**: Filtered NLTK English stopwords. 
   Justification: Removes high-frequency function words to focus analysis on semantically meaningful content.

5. **Artifact Filtering**: Excluded XML tags and OCR artifacts from token analysis. 
   Justification: OCR markup and structural tags do not represent substantive content.

---

### Hypothesis 2: Reference Dataset

6. **Source Separation**: Maintained separate analysis of reference lists vs. paragraph text. 
   Justification: Enables distinction between formal citations and active textual engagement with theorists.

7. **Named Entity Recognition**: Used spaCy's pre-trained en_core_web_sm model for PERSON extraction. 
   Justification: Robust to 1970s-1990s American English and OCR artifacts in digitized text.

8. **NER Filtering**: Applied multi-stage filtering to remove publisher names, isolated first names, and short strings. 
   Justification: Reduces noise from automated extraction while preserving legitimate person names.

9. **Manual Normalization**: Constructed lookup table mapping name variants to canonical forms. 
   Justification: Ensures accurate aggregation of multiple spelling/formatting variants for the same person.

10. **Variant Preservation**: Retained both raw and canonical name forms in output dataset. 
    Justification: Documents actual variation in source material and enables verification of normalization decisions.

---

### Hypothesis 3: Contributor Dataset

11. **Dual Source Extraction**: Separated author/creator credits from caption-extracted names. 
    Justification: Distinguishes direct authorship from being featured/discussed, revealing participation modes.

12. **Caption NER**: Applied spaCy PERSON extraction to image captions with filtering. 
    Justification: Captures visual culture contributors while reducing false positives from artwork titles and locations.

13. **Name Component Requirement**: Required both first and last name components for valid entities. 
    Justification: Filters isolated given names and initials while preserving full person identifications.

14. **Contributor Normalization**: Applied same lookup table approach as H2 for consistent methodology. 
    Justification: Standardizes spelling variants and formatting inconsistencies for reliable counting.

15. **Metadata Verification**: Confirmed all contributor records linked to valid issue and year data. 
    Justification: Ensures temporal analysis is based on complete and accurate publication information.

---

### Cross-Hypothesis Decisions

16. **Volume Segmentation**: Defined volumes as 4-issue units aligned with *Heresies*' publication cycles. 
    Justification: Structural definition ensures temporal stability and comparability across hypotheses.

17. **Missing Data Handling**: Recorded and flagged missing years; excluded incomplete records from temporal analysis. 
    Justification: Maintains analytical integrity by documenting data quality issues.

18. **CSV Output Format**: Exported all datasets in comma-separated format with UTF-8 encoding. 
    Justification: Enables reproducibility, sharing, and integration with standard analysis tools.
"""

print(cleaning_decisions)


CLEANING DECISION LOG


### Hypothesis 1: Lexical Analysis

1. **Tokenization Method**: Used NLTK's word_tokenize() to segment text into word-level tokens. 
   Justification: Preserves hyphenated compounds and contractions for domain-appropriate analysis.

2. **Lowercasing**: Applied consistent lowercasing to all tokens before lemmatization. 
   Justification: Necessary for reliable type frequency comparison without introducing conceptual difference.

3. **Lemmatization**: Applied WordNetLemmatizer to reduce inflected forms to base lemmas. 
   Justification: Groups grammatical variants for more stable frequency analysis of key concepts.

4. **Stopword Removal**: Filtered NLTK English stopwords. 
   Justification: Removes high-frequency function words to focus analysis on semantically meaningful content.

5. **Artifact Filtering**: Excluded XML tags and OCR artifacts from token analysis. 
   Justification: OCR markup and structural tags do not represent substantive content.

---

### H